In [1]:
import numpy as np
import copy
import pickle
import os

def generate_dynamic_targets(circuit_list):
    """
    Scans a list of GCAD Topo objects, extracts unique biological parts,
    and maps them to their parametric indices based on the parts library structure.
    """
    active_parts = set()
    
    for circuit in circuit_list:
        active_parts.update(circuit.part_list)
        
    targets = {}
    
    for part in active_parts:
        if part.startswith('Z'):
            targets[part] = [0, 1, 2] # Activators: 3 parameters
        elif part.startswith('I') or part.startswith('R'):
            targets[part] = [0]       # Repressors: 1 parameter
        else:
            print(f"Warning: Unknown part '{part}'. Skipping.")
            
    return targets

def generate_prior_ensemble(nominal_parts, targets, ensemble_size=200, spread_factor=2.0, dist_type='lognormal'):
    """
    Generates an ensemble of parameter models using the requested distribution.
    dist_type options: 'lognormal', 'uniform', 'u-shaped'
    """
    ensemble = []
    
    # Pre-calculate bounds based on the spread factor
    low_bound = 1.0 / spread_factor
    high_bound = spread_factor
    sigma = np.log(spread_factor) / 2.0 
    
    for _ in range(ensemble_size):
        model_copy = copy.deepcopy(nominal_parts)
        
        for part, indices in targets.items():
            if part in model_copy:
                for idx in indices:
                    # Apply the chosen mathematical distribution
                    if dist_type == 'lognormal':
                        noise_factor = np.random.lognormal(mean=0.0, sigma=sigma)
                    
                    elif dist_type == 'uniform':
                        noise_factor = np.random.uniform(low=low_bound, high=high_bound)
                    
                    elif dist_type == 'u-shaped':
                        # Beta(0.5, 0.5) creates a U-shape between 0 and 1.
                        # We scale it to perfectly fit our [low_bound, high_bound] window.
                        beta_sample = np.random.beta(a=0.5, b=0.5)
                        noise_factor = low_bound + beta_sample * (high_bound - low_bound)
                    
                    else:
                        raise ValueError(f"Unknown distribution: {dist_type}")
                        
                    model_copy[part][idx] *= noise_factor
                    
        ensemble.append(model_copy)
        
    return ensemble

# ==========================================
# 🧪 TEST BLOCK
# ==========================================
if __name__ == "__main__":
    print("--- TESTING DYNAMIC TARGETS & DISTRIBUTIONS ---\n")
    
    # 1. Load the actual textbook parts library
    parts_path = os.path.join(os.getcwd(), 'GCAD', 'parts.pkl')
    with open(parts_path, "rb") as f:
        textbook_parts = pickle.load(f)
        
    # 2. Pretend GCAD selected these circuits
    class DummyCircuit:
        def __init__(self, parts):
            self.part_list = parts
            
    selected_circuits = [
        DummyCircuit(['Z1', 'I13']), 
        DummyCircuit(['Z1', 'Z6', 'I13'])
    ]
    
    # 3. Extract Targets
    my_targets = generate_dynamic_targets(selected_circuits)
    print(f"🎯 Dynamic Targets Found: {my_targets}\n")
    
    # 4. Test all three distributions
    distributions = ['lognormal', 'uniform', 'u-shaped']
    
    for dist in distributions:
        print(f"☁️ Generating 5 models using [{dist.upper()}] prior...")
        my_ensemble = generate_prior_ensemble(
            nominal_parts=textbook_parts, 
            targets=my_targets, 
            ensemble_size=5, 
            spread_factor=2.0,
            dist_type=dist
        )
        
        # Look at the first parameter of Z1 (Nominal is 0.08)
        z1_samples = [model['Z1'][0] for model in my_ensemble]
        print(f"   Z1 [0] samples (Nominal 0.08): {[round(val, 4) for val in z1_samples]}\n")

--- TESTING DYNAMIC TARGETS & DISTRIBUTIONS ---

🎯 Dynamic Targets Found: {'Z6': [0, 1, 2], 'Z1': [0, 1, 2], 'I13': [0]}

☁️ Generating 5 models using [LOGNORMAL] prior...
   Z1 [0] samples (Nominal 0.08): [0.1256, 0.0992, 0.0953, 0.0855, 0.059]

☁️ Generating 5 models using [UNIFORM] prior...
   Z1 [0] samples (Nominal 0.08): [0.0565, 0.1167, 0.1314, 0.1018, 0.1208]

☁️ Generating 5 models using [U-SHAPED] prior...
   Z1 [0] samples (Nominal 0.08): [0.0668, 0.0577, 0.0606, 0.1568, 0.0833]

